# Round 3 — the STACCATO arm (Lever 6)

**One arm, one run.** Its control is already on disk — `data/checkpoints/r3-tupnew-stage2-best` — because that checkpoint used **this recipe, this split, this seed and the DEFAULT augmentation mix**. The only thing that differs here is the corpus: `strips_v6_stac` is `strips_v5_tupnew` re-rendered with `--staccato-noise`, i.e. **label-free dots on the notehead side**.

| | screenshot | photo | scan | corpus |
|---|---|---|---|---|
| control (`r3-tupnew-stage2-best`) | 0.65 | 0.35 | — | `strips_v5_tupnew` |
| **this arm** | 0.65 | 0.35 | — | **`strips_v6_stac`** |

⚠ **DO NOT pass `--photo-share` or `--scan-share` anywhere in this notebook.** The mix is the *control's default*, and the scan profile was settled OFF on 2026-08-19 after arm 1 came back null. A mix flag here would be a second variable.

**Why the arm exists.** `ADDED_TOKENS` has no articulation token and the renderer drew no staccato, so 0 of 40,826 strips carried one and **every dot the model had ever seen meant *longer***. Measured with a paired control: **72.7%** of staccato-bearing strips get an augmentation dot the gold does not have, against **0.0%** on the identical music unmarked. What the dots teach is POSITIONAL — a dot lengthens only when it sits BESIDE the notehead.

**The floors are already signed** (`docs/rung3/levers.md` Lever 6) and are not re-opened: (1) primary — the false-dot rate must fall from 72.7%; (2) no-regression on real dots, **EASY+MID only**, hard tier reported and never gated; (3) pitch/AEU macro F1 reported, so the price of clause 1 is on the record.

⚠ **The two manifests are byte-identical.** The dots are label-free and `staccatoseed` is not a manifest field, which is what makes the result attributable to the pixels. The render came out 15 strips larger than the control's and **the dots were not the cause** — it reproduced with the flag off — so the manifest was filtered back to the control's exact row set (see `README-yield-drift.md` beside the corpus).

⚠ **Exam strips are not on this VM and the exam is not read here.** One shot, later, on Round 3's final model.

In [ ]:
# ===== THE ONLY KNOBS IN THIS NOTEBOOK — and the mix is NOT one of them =====
ARM = 'stac'
STRIPS = 'data/synthetic/strips_v6_stac'   # the arm's corpus: tupnew + --staccato-noise
ZIP = 'tnc_round3_stac_colab.zip'
DRIVE = '/content/drive/MyDrive/tnc'
# No PHOTO_SHARE / SCAN_SHARE: this arm trains at train.py's DEFAULT mix, which is what the control
# trained at. Adding a mix flag here would make the run unattributable (see the header).
print(ARM, STRIPS, ZIP, 'mix: train.py defaults (screenshot 0.65 / photo 0.35 / scan off)')

In [ ]:
# Which GPU did we get? (T4 16GB / L4 24GB / A100 40GB)
!nvidia-smi

In [ ]:
# Mount Google Drive (approve the popup).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%time
# Copy the package Drive -> VM disk and unzip (fast local disk for the dataloader).
!cp {DRIVE}/{ZIP} /content/
!rm -rf /content/tnc && mkdir /content/tnc
!cd /content/tnc && unzip -q /content/{ZIP}

# WHICH CORPUS IS ACTUALLY ON DISK. The dots are pixels only, so the manifest cannot tell this
# corpus from its control — render_config.json is the ONLY place the arm is checkable, which is why
# it is asserted here as well as in make_round3_colab_zip.sh.
import json
cfg = json.load(open(f'/content/tnc/{STRIPS}/render_config.json'))
print(cfg)
assert cfg['staccatoNoise'] is True, f'{ZIP} is NOT the staccato corpus — this is the control'
assert cfg['legacyTupletMark'] is False and cfg['thinSharps'] is True and cfg['printNoise'] is False
assert cfg.get('concaveTuplet', False) is False, 'the concave tuplet mark belongs to the FINAL render, not an arm'
!wc -l /content/tnc/{STRIPS}/manifest.jsonl   # expect 40826 — the control's exact row count
!python -c "import json;s=json.load(open('/content/tnc/data/split_v4.json'));print('train',len(s['train_pieces']),'val',len(s['val_pieces']))"

In [ ]:
# Dependencies (torch + torchvision are preinstalled on Colab).
!pip -q install transformers albumentations opencv-python-headless

In [ ]:
# ===== THE DOTS ARE THE EXPERIMENT — prove they are in the pixels before spending a GPU hour =====
# Nothing downstream records them: the labels, the manifest and the split are identical to the
# control's by design. So the check is on the IMAGES, and on the mix being the control's default.
%cd /content/tnc
import sys
sys.path.insert(0, 'src/vision')
from augment import Augmenter
a = Augmenter(seed=7)
assert (a.photo_share, a.scan_share) == (0.35, 0.0), (a.photo_share, a.scan_share)
print(f'mix: screenshot {1-a.photo_share-a.scan_share:.2f} / photo {a.photo_share} / scan {a.scan_share}  (control default)')

# The dots are drawn on ALREADY-DOTTED notes and in short runs, so they are sparse: sample strips
# whose label carries a duration dot, where the draw is most likely to have fired.
import json, random
rows = [json.loads(l) for l in open(f'{STRIPS}/manifest.jsonl')]
dotted = [r for r in rows if '.' in r['label']]
print(f'{len(rows)} strips, {len(dotted)} carrying a dotted duration')
random.Random(7).shuffle(dotted)
from PIL import Image
import numpy as np
inks = [np.asarray(Image.open(f"{STRIPS}/{r['image']}").convert('L')).mean() for r in dotted[:200]]
print('mean grey over 200 dotted-label strips:', round(float(np.mean(inks)), 3))
display(Image.open(f"{STRIPS}/{dotted[0]['image']}"))   # LOOK: dots above/below noteheads, not beside

In [ ]:
# SHAKEOUT (~3 min): 150 tiny steps from BASE — a WIRING smoke, not a result.
# Expect: `vocab: +25 tokens -> 100 ids`, the three real pools listed, `exam-disjointness OK`,
# `augment=on (screenshot 0.65 / photo 0.35)`, and val loss FALLING.
%cd /content/tnc
!python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir data/real/rung3/strips_nota --real-dir data/real/rung3/strips_r1 \
    --real-dir data/real/rung3/strips_tup \
    --every-share 0.15 --out-dir /content/r3-shakeout \
    --lr 3e-5 --warmup-steps 30 --max-steps 150 --batch-size 8 \
    --limit-val 40 --eval-every 50 --save-every 50 --log-every 25 --num-workers 2

In [ ]:
# ===== CALIBRATE THROUGHPUT ON *THIS* RUNTIME (~2-3 min) — before any long run =====
#   hours = (steps * batch) / samples_per_sec / 3600
# ⚠ This arm's augmentation is the CONTROL's, so throughput should read like the tupnew run's — the
# dots are baked into the PNGs, not applied per sample. A big slowdown here means the runtime, not
# the arm; raise --num-workers before raising anything else.
# ⚠ Whatever you set, the STEP COUNTS AND BATCH SIZE must match the control's: 6000 @ 16, then
# 2000 @ 16. --num-workers and the GPU model do not change the result; those two do.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!nproc
%cd /content/tnc
!python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir data/real/rung3/strips_nota --real-dir data/real/rung3/strips_r1 \
    --real-dir data/real/rung3/strips_tup \
    --every-share 0.15 --out-dir /content/calib \
    --lr 3e-5 --warmup-steps 20 --max-steps 60 --batch-size 16 \
    --limit-val 8 --eval-every 60 --save-every 60 --log-every 20 --num-workers 10

In [ ]:
# ===== STAGE 1 — carry-dominant SYNTHETIC ONLY, from BASE =====
# No --real-dir: this builds the carry-native synthetic checkpoint stage 2 specialises. It is also
# where the dots do their work, because real strips train CLEAN (--augment-real is
# off, and stays off — double-degrading a blurry nota scan buries its signal).
# ⚠ `-u` is not cosmetic: Colab block-buffers a subprocess's stdout, so without it the log
# lines sit in an 8 KB buffer and a healthy run looks frozen for minutes at a time.
# Recipe identical to the control's, INCLUDING the mix — the corpus is the only difference.
%cd /content/tnc
!python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --every-share 0.15 --out-dir {DRIVE}/r3-{ARM}-stage1 \
    --lr 3e-5 --max-steps 6000 --batch-size 16 --num-workers 10  # ~nproc-2; T4 (2 vCPU) use 2

In [ ]:
# ===== STAGE 2 — real-SPECIALISATION fine-tune from stage 1 =====
# Fresh LOW lr + short warmup from the stage-1 checkpoint. `:9` as in the control — the suffix
# oversamples each real pool so real is ~1/3 of batches. It must be THE SAME AS THE CONTROL'S.
#
# Selection caveat carried over: oversampled real overfits fast and `best` is picked on a
# synth-dominated val mix — so both `best` and `last` come home. ⚠ The control on disk is
# `r3-tupnew-stage2-BEST`, so `best` is the comparison and `last` is reported beside it.
%cd /content/tnc
!python -u src/vision/train.py --model {DRIVE}/r3-{ARM}-stage1/best \
    --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir data/real/rung3/strips_nota:9 \
    --real-dir data/real/rung3/strips_r1:9 \
    --real-dir data/real/rung3/strips_tup:9 \
    --every-share 0.15 --out-dir {DRIVE}/r3-{ARM}-stage2 \
    --lr 1e-5 --warmup-steps 100 --max-steps 2000 --batch-size 16 --num-workers 10

In [ ]:
# RESUME after a disconnect: re-run the setup cells, then this with the SAME flags as the stage
# you were running (edit out-dir/flags to match). --resume reloads model+optimizer+scheduler from
# <out-dir>/last and ignores --model.
%cd /content/tnc
!python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --every-share 0.15 --out-dir {DRIVE}/r3-{ARM}-stage1 \
    --lr 3e-5 --max-steps 6000 --batch-size 16 --num-workers 10 --resume

In [ ]:
# ===== SANITY ONLY — did anything break? =====
# NOT the pre-registered number. That is read on the Mac, on the tier/medium pools, paired against
# the control. This cell exists so a broken run is caught before it is downloaded, and to see
# `best` and `last` side by side.
%cd /content/tnc
!python src/vision/make_realval_pool.py --real-dir data/real/rung3/strips_nota \
    --real-dir data/real/rung3/strips_r1 --real-dir data/real/rung3/strips_tup \
    --split data/split_v4.json

for ck in [f'r3-{ARM}-stage2/best', f'r3-{ARM}-stage2/last']:
    print('=' * 70, '\n==', ck)
    !python src/vision/eval_omr.py --checkpoint {DRIVE}/{ck} \
        --strips-dir data/real/rung3/_realval --split none --show-errors 0

## After the run

1. **Download the stage-2 checkpoints** from `MyDrive/tnc/r3-stac-stage2/` into `data/checkpoints/` on the Mac. **Both** `best` and `last` — stage 2 saves `best` on a synth-dominated val mix, which is the selector Lever 5 already distrusts, and arm 1 showed the two can disagree.
2. **Read the PRIMARY number — the false-dot rate — on the paired pools:**
   ```bash
   .venv-ml/bin/python src/vision/eval_omr.py --checkpoint data/checkpoints/r3-stac-stage2-best \
       --strips-dir data/real/rung3/_staccato_falsedot_stac --split none
   .venv-ml/bin/python src/vision/eval_omr.py --checkpoint data/checkpoints/r3-stac-stage2-best \
       --strips-dir data/real/rung3/_staccato_falsedot_ctl --split none
   ```
   Against the **72.7% / 0.0%** baseline in `docs/METRICS-DIAGNOSTICS.md`. Do not re-derive the baseline from memory.
3. **Clause 2 — no-regression on REAL dots, EASY+MID only.** Hard tier is reported, never gated: ~12 real-dot instances in total and the least reliable gold we own (settled 2026-08-19, before training).
4. **Clause 3 — report pitch/AEU macro F1**, so the price of clause 1 is on the record. Paired against the control:
   ```bash
   .venv-ml/bin/python scripts/rung3/paired_arm_score.py \
       --ctl data/checkpoints/r3-tupnew-stage2-best --arm data/checkpoints/r3-stac-stage2-best \
       --pool data/real/rung3/_realval_v2 --out data/real/rung3/lever6/stac_best.json
   ```
5. ⚠ **Watch the slur distractor's precedent**: it bought `\tup3` precision 15.1% → 91.2% and cost recall 92.7% → 83.8%, below its own floor. Clause 2 exists so that shape of outcome is caught, not discovered later.
6. **A null is written up as a null**, and `STACCATO_RATE` is not re-tuned to chase a win. It is chosen, not measured — replacing it means counting staccato frequency in real editions.
7. **The exam is still unread.** One shot, on Round 3's final model, against `docs/rung3/round3-criteria.md`.